In [1]:
# ============================================================
# CELL 1 — IMPORTS & CONFIG
# ============================================================

import os
import sys
import gc
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Repository
# ------------------------------------------------------------

REPO_PATH = r"C:\Users\Anusha\engression\engression-ts"

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

SEED = 42

INPUT_CHUNK_LENGTH = 24
OUTPUT_CHUNK_LENGTH = 24

BATCH_SIZE = 64

# Engression training samples
NUM_SAMPLES_TRAIN = 2

# Prediction samples
NUM_SAMPLES_PRED = 100

NOISE_STD = 1.0
NOISE_TYPE = "uniform"

LEARNING_RATE = 1e-3

# 30 epochs at batch_size=64
MAX_STEPS = 109

DATASET_NAME = "solar_nips"

RESULTS_FILE = "nf_solar_results.csv"

print("Repository:", REPO_PATH)
print("Dataset:", DATASET_NAME)
print("Batch size:", BATCH_SIZE)
print("Training samples:", NUM_SAMPLES_TRAIN)
print("Prediction samples:", NUM_SAMPLES_PRED)
print("Max steps:", MAX_STEPS)

Repository: C:\Users\Anusha\engression\engression-ts
Dataset: solar_nips
Batch size: 64
Training samples: 2
Prediction samples: 100
Max steps: 109


In [2]:
# ============================================================
# CELL 2 — DETERMINISTIC SEEDS
# ============================================================

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(
    True,
    warn_only=True
)

pl.seed_everything(SEED, workers=True)

print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Seed set to 42


Seed: 42
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
# ============================================================
# CELL 3 — LOAD SOLAR NIPS
# ============================================================

from gluonts.dataset.repository.datasets import get_dataset
from gluonts.dataset.multivariate_grouper import MultivariateGrouper

ds = get_dataset(
    DATASET_NAME,
    regenerate=False
)

train_list = list(ds.train)
test_list = list(ds.test)

freq = ds.metadata.freq

num_test_dates = len(test_list) // len(train_list)

target_dim = int(
    ds.metadata.feat_static_cat[0].cardinality
)

print("Dataset:", DATASET_NAME)
print("Training series:", len(train_list))
print("Target dimensions:", target_dim)
print("Test windows:", num_test_dates)
print("Frequency:", freq)

Dataset: solar_nips
Training series: 137
Target dimensions: 137
Test windows: 7
Frequency: h


In [3]:
# ============================================================
# CELL 4 — GLUONTS → DARTS
# ============================================================

def gluonts_item_to_darts_mv(item, freq):

    start = (
        item["start"].to_timestamp()
        if hasattr(item["start"], "to_timestamp")
        else pd.Timestamp(item["start"])
    )

    target = np.asarray(item["target"])

    if target.ndim != 2:
        raise ValueError(
            f"Expected multivariate target, got {target.shape}"
        )

    # GluonTS: (D, T)
    # Darts:   (T, D)
    values = target.T

    times = pd.date_range(
        start=start,
        periods=values.shape[0],
        freq=freq
    )

    columns = [
        f"node_{i}"
        for i in range(values.shape[1])
    ]

    return TimeSeries.from_times_and_values(
        times,
        values,
        columns=columns
    )


train_grouper = MultivariateGrouper(
    max_target_dim=target_dim
)

test_grouper = MultivariateGrouper(
    num_test_dates=num_test_dates,
    max_target_dim=target_dim
)

train_mv_items = list(
    train_grouper(train_list)
)

test_mv_items = list(
    test_grouper(test_list)
)

train_ts = gluonts_item_to_darts_mv(
    train_mv_items[0],
    freq
)

test_ts_list = [
    gluonts_item_to_darts_mv(
        item,
        freq
    )
    for item in test_mv_items
]

print("Training shape:", train_ts.shape)
print("Test windows:", len(test_ts_list))

Training shape: (7009, 137, 1)
Test windows: 7


In [4]:
# ============================================================
# CELL 5 — SCALE TARGET
# ============================================================

y_scaler = Scaler()

train_y_sc = y_scaler.fit_transform(
    train_ts
).astype(np.float32)

print("Scaled training shape:", train_y_sc.shape)

Scaled training shape: (7009, 137, 1)


In [5]:
# ============================================================
# CELL 6 — CREATE TEST WINDOWS
# ============================================================

def get_test_windows(dataset, num_nodes):

    all_series = []

    for entry in dataset:

        idx = pd.date_range(
            start=entry["start"].to_timestamp(),
            periods=len(entry["target"]),
            freq=entry["start"].freqstr
        )

        all_series.append(
            pd.Series(
                entry["target"],
                index=idx
            )
        )

    num_windows = len(all_series) // num_nodes

    windows = []

    for w in range(num_windows):

        start_idx = w * num_nodes
        end_idx = (w + 1) * num_nodes

        window_df = pd.concat(
            all_series[start_idx:end_idx],
            axis=1
        )

        window_df.columns = [
            f"node_{i}"
            for i in range(num_nodes)
        ]

        windows.append(window_df)

    return windows


test_windows = get_test_windows(
    ds.test,
    num_nodes=target_dim
)

print("Number of test windows:", len(test_windows))

for i, window in enumerate(test_windows):
    print(
        f"Window {i+1}:",
        window.shape,
        window.index[0],
        "→",
        window.index[-1]
    )

Number of test windows: 7
Window 1: (7033, 137) 2006-01-01 00:00:00 → 2006-10-21 00:00:00
Window 2: (7057, 137) 2006-01-01 00:00:00 → 2006-10-22 00:00:00
Window 3: (7081, 137) 2006-01-01 00:00:00 → 2006-10-23 00:00:00
Window 4: (7105, 137) 2006-01-01 00:00:00 → 2006-10-24 00:00:00
Window 5: (7129, 137) 2006-01-01 00:00:00 → 2006-10-25 00:00:00
Window 6: (7153, 137) 2006-01-01 00:00:00 → 2006-10-26 00:00:00
Window 7: (7177, 137) 2006-01-01 00:00:00 → 2006-10-27 00:00:00


In [6]:
# ============================================================
# CELL 7 — METRICS
# ============================================================

def to_tensor(x):
    if isinstance(x, torch.Tensor):
        return x.float()
    return torch.as_tensor(
        x,
        dtype=torch.float32
    )


def point_forecast(y_pred, method="median"):

    y_pred = to_tensor(y_pred)

    if method == "mean":
        return torch.mean(y_pred, dim=-1)

    if method == "median":
        return torch.median(y_pred, dim=-1).values

    if isinstance(method, float):
        return torch.quantile(
            y_pred,
            method,
            dim=-1
        )

    raise ValueError(
        "point_method must be mean, median or quantile"
    )


def mae(y_true, y_pred):
    return torch.mean(
        torch.abs(
            to_tensor(y_true) -
            point_forecast(y_pred)
        )
    ).item()


def mse(y_true, y_pred):
    return torch.mean(
        (
            to_tensor(y_true) -
            point_forecast(y_pred)
        ) ** 2
    ).item()


def rmse(y_true, y_pred):
    return np.sqrt(
        mse(y_true, y_pred)
    )


def smape(y_true, y_pred, eps=1e-8):

    y_true = to_tensor(y_true)
    pred = point_forecast(y_pred)

    denominator = torch.clamp(
        (torch.abs(y_true) + torch.abs(pred)) / 2,
        min=eps
    )

    return torch.mean(
        torch.abs(y_true - pred) / denominator * 100
    ).item()


def mape(y_true, y_pred, eps=1e-8):

    y_true = to_tensor(y_true)
    pred = point_forecast(y_pred)

    denominator = torch.clamp(
        torch.abs(y_true),
        min=eps
    )

    return torch.mean(
        torch.abs(y_true - pred) /
        denominator * 100
    ).item()


def mpe(y_true, y_pred, eps=1e-8):

    y_true = to_tensor(y_true)
    pred = point_forecast(y_pred)

    denominator = torch.clamp(
        y_true,
        min=eps
    )

    return torch.mean(
        (y_true - pred) /
        denominator * 100
    ).item()


def smdape(y_true, y_pred, eps=1e-8):

    y_true = to_tensor(y_true)
    pred = point_forecast(y_pred)

    denominator = torch.clamp(
        (torch.abs(y_true) + torch.abs(pred)) / 2,
        min=eps
    )

    error = (
        torch.abs(y_true - pred) /
        denominator * 100
    )

    return torch.median(error).item()


def opl(y_true, y_pred):

    y_true = to_tensor(y_true)
    pred = point_forecast(y_pred)

    true_direction = torch.sign(
        y_true[1:] - y_true[:-1]
    )

    pred_direction = torch.sign(
        pred[1:] - y_true[:-1]
    )

    return torch.mean(
        torch.abs(
            true_direction -
            pred_direction
        ) / 2
    ).item()


def mase(y_true, y_pred, y_train):

    y_true = to_tensor(y_true)
    y_train = to_tensor(y_train)

    pred = point_forecast(y_pred)

    forecast_error = torch.abs(
        y_true - pred
    )

    scale = torch.mean(
        torch.abs(
            y_train[1:] -
            y_train[:-1]
        ),
        dim=0
    )

    scale = torch.clamp(
        scale,
        min=1e-8
    )

    return torch.mean(
        forecast_error /
        scale.unsqueeze(0)
    ).item()


def rmsse(y_true, y_pred, y_train):

    y_true = to_tensor(y_true)
    y_train = to_tensor(y_train)

    pred = point_forecast(y_pred)

    forecast_error = (
        y_true - pred
    ) ** 2

    scale = torch.mean(
        (
            y_train[1:] -
            y_train[:-1]
        ) ** 2,
        dim=0
    )

    scale = torch.clamp(
        scale,
        min=1e-8
    )

    return torch.mean(
        torch.sqrt(
            forecast_error /
            scale.unsqueeze(0)
        )
    ).item()


def crps(y_true, y_pred):

    y_true = to_tensor(y_true)
    y_pred = to_tensor(y_pred)

    y_true = y_true.unsqueeze(-1)

    term1 = torch.mean(
        torch.abs(
            y_pred - y_true
        ),
        dim=-1
    )

    pred_i = y_pred.unsqueeze(-1)
    pred_j = y_pred.unsqueeze(-2)

    term2 = torch.mean(
        torch.abs(pred_i - pred_j),
        dim=(-1, -2)
    )

    return torch.mean(
        term1 - 0.5 * term2
    ).item()


def picp(y_true, y_pred, alpha=0.05):

    y_true = to_tensor(y_true)
    y_pred = to_tensor(y_pred)

    lower = torch.quantile(
        y_pred,
        alpha / 2,
        dim=-1
    )

    upper = torch.quantile(
        y_pred,
        1 - alpha / 2,
        dim=-1
    )

    return torch.mean(
        (
            (y_true >= lower) &
            (y_true <= upper)
        ).float()
    ).item()


def mpiw(y_pred, alpha=0.05):

    y_pred = to_tensor(y_pred)

    lower = torch.quantile(
        y_pred,
        alpha / 2,
        dim=-1
    )

    upper = torch.quantile(
        y_pred,
        1 - alpha / 2,
        dim=-1
    )

    return torch.mean(
        upper - lower
    ).item()


def mis(y_true, y_pred, alpha=0.05):

    y_true = to_tensor(y_true)
    y_pred = to_tensor(y_pred)

    lower = torch.quantile(
        y_pred,
        alpha / 2,
        dim=-1
    )

    upper = torch.quantile(
        y_pred,
        1 - alpha / 2,
        dim=-1
    )

    width = upper - lower

    penalty_lower = (
        2 / alpha
    ) * torch.maximum(
        lower - y_true,
        torch.zeros_like(y_true)
    )

    penalty_upper = (
        2 / alpha
    ) * torch.maximum(
        y_true - upper,
        torch.zeros_like(y_true)
    )

    return torch.mean(
        width +
        penalty_lower +
        penalty_upper
    ).item()

In [7]:
# ============================================================
# CELL 8 — EVALUATION
# ============================================================

def evaluate_model(
    model,
    test_windows,
    y_scaler,
    pred_len=24,
    num_samples=100,
    seed=42,
):

    window_results = []
    total_inference_time = 0.0

    for window_idx, window_df in enumerate(test_windows):

        print(
            f"  Evaluating window "
            f"{window_idx + 1}/{len(test_windows)}..."
        )

        full_ts = TimeSeries.from_dataframe(
            window_df
        ).astype(np.float32)

        full_sc = y_scaler.transform(
            full_ts
        ).astype(np.float32)

        past_sc = full_sc[:-pred_len]
        future = full_ts[-pred_len:]

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        start = time.time()

        forecast_sc = model.predict(
            n=pred_len,
            series=past_sc,
            num_samples=num_samples,
            clip_preds=True,
            verbose=False,
            random_state=seed,
        )

        inference_time = time.time() - start
        total_inference_time += inference_time

        # ----------------------------------------------------
        # Back to original scale
        # ----------------------------------------------------

        forecast = y_scaler.inverse_transform(
            forecast_sc
        )

        assert forecast.time_index.equals(
            future.time_index
        ), f"Time mismatch in window {window_idx}"

        # Shape:
        # forecast -> (T, D, S)
        y_pred = forecast.all_values(
            copy=False
        )

        # Ground truth -> (T, D)
        y_true = future.all_values(
            copy=False
        ).squeeze(-1)

        # Training history for scaled metrics
        y_train = full_ts[:-pred_len].all_values(
            copy=False
        ).squeeze(-1)

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        metrics = {

            # Point
            "MAE": mae(y_true, y_pred),
            "MSE": mse(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
            "sMdAPE": smdape(y_true, y_pred),
            "MPE": mpe(y_true, y_pred),
            "OPL": opl(y_true, y_pred),

            # Scaled
            "MASE": mase(
                y_true,
                y_pred,
                y_train
            ),

            "RMSSE": rmsse(
                y_true,
                y_pred,
                y_train
            ),

            # Probabilistic
            "CRPS": crps(
                y_true,
                y_pred
            ),

            "PICP": picp(
                y_true,
                y_pred,
                alpha=0.05
            ),

            "MIS": mis(
                y_true,
                y_pred,
                alpha=0.05
            ),

            "MPIW": mpiw(
                y_pred,
                alpha=0.05
            ),

        }

        window_results.append(metrics)

    # --------------------------------------------------------
    # Average across all 7 windows
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        window_results
    )

    summary = results_df.mean().to_frame(
        "Average_Score"
    )

    summary.loc[
        "Inference Time",
        "Average_Score"
    ] = (
        total_inference_time /
        len(test_windows)
    )

    return summary

In [9]:
# ============================================================
# CELL 9 — HEALTHY MODELS
# ============================================================

from engressionts.models.neuralforecast import (
    EnBiTCN,
    EniTransformer,
    EnKAN,
    EnMLP,
    EnMLPMultivariate,
    EnPatchTST,
    EnRMoK,
    EnSOFTS,
    EnSOFTSSharp,
    EnStemGNN,
    EnTimeXer,
    EnTSMixerx,
    EnXLinear,
)

MODELS_TO_RUN = [

    ("EnBiTCN", EnBiTCN),
    ("EniTransformer", EniTransformer),
    ("EnKAN", EnKAN),
    ("EnMLP", EnMLP),
    ("EnMLPMultivariate", EnMLPMultivariate),
    ("EnPatchTST", EnPatchTST),
    ("EnRMoK", EnRMoK),
    ("EnSOFTS", EnSOFTS),
    ("EnSOFTSSharp", EnSOFTSSharp),
    ("EnStemGNN", EnStemGNN),
    ("EnTimeXer", EnTimeXer),
    ("EnTSMixerx", EnTSMixerx),
    ("EnXLinear", EnXLinear),
]

print("Models to run:")

for name, _ in MODELS_TO_RUN:
    print(" -", name)

Models to run:
 - EnBiTCN
 - EniTransformer
 - EnKAN
 - EnMLP
 - EnMLPMultivariate
 - EnPatchTST
 - EnRMoK
 - EnSOFTS
 - EnSOFTSSharp
 - EnStemGNN
 - EnTimeXer
 - EnTSMixerx
 - EnXLinear


In [10]:
# ============================================================
# CELL 10 — MODELS NOT RUNNING IN THIS BENCHMARK
# ============================================================

# MEMORY-LIMITED
#
# EnAutoformer
# EnInformer
# EnTimesNet
# EnxLSTM
#
# Known configuration:
#
# batch_size = 8
# max_steps = 26130
#
# ------------------------------------------------------------
#
# NUMERICAL STABILITY TODO
#
# EnFEDformer
#
# Currently excluded because of the known NaN issue.
#
# ------------------------------------------------------------
#
# NOT A STANDARD MODEL FOR THIS BENCHMARK
#
# EnHINT
#
# Wrapper / hierarchical model.
# Not applicable to this flat Solar benchmark.
#
# ------------------------------------------------------------
#
# REMOVED
#
# EnDeepAR
# EnDeepNPTS
# EnVanillaTransformer
#
# ------------------------------------------------------------

print("Heavy / TODO models are intentionally excluded.")

Heavy / TODO models are intentionally excluded.


In [10]:
# ============================================================
# CELL 11 — TRAIN + EVALUATE ALL MODELS
# ============================================================

from engressionts.models.darts.en_nf_model import (
    EngressionNeuralForecastModel
)

all_results = []

for model_idx, (model_name, model_cls) in enumerate(
    MODELS_TO_RUN,
    start=1
):

    print("\n")
    print("=" * 70)
    print(
        f"[{model_idx}/{len(MODELS_TO_RUN)}] "
        f"{model_name}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Clean memory
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = EngressionNeuralForecastModel(

        input_chunk_length=INPUT_CHUNK_LENGTH,

        output_chunk_length=OUTPUT_CHUNK_LENGTH,

        model=model_cls,

        batch_size=BATCH_SIZE,

        model_kwargs={
            "num_samples_train": NUM_SAMPLES_TRAIN,
            "noise_std": NOISE_STD,
            "noise_type": NOISE_TYPE,
        },

        random_state=SEED,

        optimizer_kwargs={
            "lr": LEARNING_RATE
        },

        pl_trainer_kwargs={
            "accelerator": "gpu",
            "devices": [0],
            "max_steps": MAX_STEPS,
        },
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    print("\nTraining...")

    start = time.time()

    try:

        model.fit(
            series=train_y_sc,
            verbose=True,
        )

        training_time = time.time() - start

        print(
            f"\nTraining time: "
            f"{training_time:.2f} seconds"
        )

    except Exception as e:

        print(
            f"\n❌ {model_name} FAILED DURING TRAINING"
        )

        print(
            type(e).__name__,
            ":",
            e
        )

        continue

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    print("\nEvaluating 7 test windows...")

    try:

        metrics = evaluate_model(
            model=model,
            test_windows=test_windows,
            y_scaler=y_scaler,
            pred_len=OUTPUT_CHUNK_LENGTH,
            num_samples=NUM_SAMPLES_PRED,
            seed=SEED,
        )

        print("\nResults:")
        display(metrics)

    except Exception as e:

        print(
            f"\n❌ {model_name} FAILED DURING EVALUATION"
        )

        print(
            type(e).__name__,
            ":",
            e
        )

        continue

    # --------------------------------------------------------
    # Save one row
    # --------------------------------------------------------

    row = {
        "Model": model_name,

        "Batch Size": BATCH_SIZE,

        "Max Steps": MAX_STEPS,

        "Training Samples": NUM_SAMPLES_TRAIN,

        "Prediction Samples": NUM_SAMPLES_PRED,

        "Noise Std": NOISE_STD,

        "Noise Type": NOISE_TYPE,

        "Input Length": INPUT_CHUNK_LENGTH,

        "Forecast Horizon": OUTPUT_CHUNK_LENGTH,

        "Training Time (s)": training_time,

    }

    for metric_name in metrics.index:

        row[metric_name] = metrics.loc[
            metric_name,
            "Average_Score"
        ]

    all_results.append(row)

    # --------------------------------------------------------
    # Save after EVERY model
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        all_results
    )

    results_df.to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"\n✓ Saved results to {RESULTS_FILE}"
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True




[1/13] EnBiTCN

Training...


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA GeForce RTX 3050 6GB Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | trai

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

  Evaluating window 6/7...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Evaluating window 7/7...

Results:


,Average_Score
MAE,1.868204e+01
MSE,1.259299e+03
RMSE,3.311875e+01
MAPE,2.101238e+10
sMAPE,8.821531e+01
sMdAPE,5.981172e+01
MPE,-2.101238e+10
OPL,2.977740e-01
MASE,1.475442e+00
RMSSE,8.700156e-01



✓ Saved results to nf_solar_results.csv


In [ ]:
# ============================================================
# CELL 12 — FINAL RESULTS
# ============================================================

if len(all_results) == 0:

    print(
        "No models completed successfully."
    )

else:

    final_results = pd.DataFrame(
        all_results
    )

    print("=" * 70)
    print("FINAL NF ENGRESSION RESULTS")
    print("=" * 70)

    display(
        final_results
    )

    final_results.to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"\nSaved to: {RESULTS_FILE}"
    )

FINAL NF ENGRESSION RESULTS


,Model,Batch Size,Max Steps,Training Samples,Prediction Samples,Noise Std,Noise Type,Input Length,Forecast Horizon,Training Time (s),...,sMdAPE,MPE,OPL,MASE,RMSSE,CRPS,PICP,MIS,MPIW,Inference Time
0,EnBiTCN,64,109,2,100,1.0,uniform,24,24,15.964740,...,185.618353,-9.169232e+10,0.415424,2.200044,1.298013,22.013250,0.612921,431.987849,47.686978,0.091664
1,EniTransformer,64,109,2,100,1.0,uniform,24,24,48.798908,...,67.691872,-2.854218e+10,0.296391,1.456538,0.858631,14.264869,0.867136,199.930835,64.733867,0.968641
2,EnKAN,64,109,2,100,1.0,uniform,24,24,63.455391,...,138.998268,-3.466427e+10,0.386453,1.584593,0.934639,14.439596,0.922402,171.047873,68.593126,1.148329
3,EnMLP,64,109,2,100,1.0,uniform,24,24,14.363990,...,96.000349,-1.078987e+10,0.305459,1.407635,0.829919,13.006954,0.938434,143.990223,75.154567,0.132941
4,EnMLPMultivariate,64,109,2,100,1.0,uniform,24,24,3.263511,...,70.529932,-1.132213e+10,0.292900,1.309367,0.771661,13.080855,0.742961,238.946469,33.077266,0.131220
5,EnPatchTST,64,109,2,100,1.0,uniform,24,24,83.720216,...,62.880388,-2.227372e+10,0.296210,1.514176,0.892669,14.786047,0.928528,170.751835,86.386512,0.776323
6,EnRMoK,64,109,2,100,1.0,uniform,24,24,20.057209,...,132.882627,-3.704255e+10,0.363467,1.633817,0.963344,15.426163,0.928137,180.691644,82.149858,0.108521
7,EnSOFTS,64,109,2,100,1.0,uniform,24,24,47.715155,...,54.565085,-2.069330e+10,0.254726,1.344960,0.792762,13.203680,0.867397,165.369020,63.290043,0.196130
8,EnSOFTSSharp,64,109,2,100,1.0,uniform,24,24,46.587390,...,41.721324,-1.563766e+10,0.246634,1.336075,0.787648,13.179879,0.864181,172.770691,62.388918,0.309936
9,EnStemGNN,64,109,2,100,1.0,uniform,24,24,43.152930,...,69.867896,-3.224241e+10,0.328535,1.668816,0.984521,18.076334,0.555613,484.145011,25.303036,0.196002



Saved to: nf_solar_results.csv


: 